# Lecture 3 Lab: From Raw Table to Model-Ready Data
**BINF 6210/8210 - Machine Learning for Bioinformatics**

Learning goals: audit structure and quality; reason about missingness; encode and scale features; preserve a clean train/test boundary.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


In [ ]:
n = 240
df = pd.DataFrame({
    "sample_id": [f"S{i:03d}" for i in range(n)],
    "age": rng.normal(52, 14, n).clip(18, 85),
    "bmi": rng.normal(27, 5, n).clip(16, 48),
    "batch": rng.choice(["B1", "B2", "B3"], n, p=[.45,.35,.20]),
    "sex": rng.choice(["Female", "Male"], n),
    "metabolite_a": rng.lognormal(1.2, .8, n),
    "metabolite_b": rng.lognormal(.5, .6, n),
})
logit = -3 + .045*df.age + .7*np.log1p(df.metabolite_a) + .35*(df.sex=="Male")
df["case"] = rng.binomial(1, 1/(1+np.exp(-logit)))
for col, frac in {"bmi":.08,"metabolite_a":.12,"batch":.04}.items():
    df.loc[rng.choice(n, int(n*frac), replace=False), col] = np.nan
df.loc[5, "bmi"] = 120  # deliberate data-quality issue
df.head()


## 1. Audit before changing anything

Ask: What is one row? What is the prediction target? Which columns are identifiers, features, outcomes, or metadata?


In [ ]:
print(df.shape)
display(df.dtypes.to_frame("dtype"))
display(df.isna().mean().sort_values(ascending=False).to_frame("missing_fraction"))
display(df.describe(include="all").T)


### Pair activity
Identify at least four concerns. For each, write whether it is a **valid value**, **measurement issue**, **missingness issue**, or **modeling decision**.


In [ ]:
# Your notes here
quality_notes = []
quality_notes


## 2. Missing values are data, not merely empty cells

Median/mode imputation is a defensible baseline, but it does not recover the unknown truth. The strategy must match the scientific mechanism and prediction setting.


In [ ]:
df.groupby("case")[["bmi","metabolite_a"]].agg(["count","median","mean"])


## 3. Split before learning preprocessing parameters


In [ ]:
X = df.drop(columns=["case","sample_id"])
y = df["case"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.25,stratify=y,random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())


## 4. Match transformations to feature types


In [ ]:
numeric = ["age","bmi","metabolite_a","metabolite_b"]
categorical = ["batch","sex"]
num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),("scale", StandardScaler())])
cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),("encode", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("num",num_pipe,numeric),("cat",cat_pipe,categorical)])
Xt = preprocess.fit_transform(X_train)
print("Transformed training shape:", Xt.shape)
print(preprocess.get_feature_names_out())


## 5. Exit ticket

1. Which objects learned parameters from the training data?
2. Why is `sample_id` excluded?
3. What would happen if a new batch label appeared in the test set?
